# Notebook 7: Every table and figure for the paper

**Runtime:** a normal **CPU** runtime is enough (no GPU needed). About 10–15 minutes.

This notebook does not train or score anything new except one small speed test. It **re-reads the saved result files** of Notebooks 1–8c and turns them into the paper's tables and figures, so every number in the paper comes from a file, not from memory or copy-paste.

**Output** (in Drive, `results/paper/`):
- `tables/`: each table as `.csv` (to read) and `.tex` (to paste into Overleaf)
- `figures/`: each figure as `.pdf` (for the paper) and `.png` (to look at)
- `numbers.json`: every number the text quotes, in one place. When we write a sentence with a number, it must match this file.

**Also new here:**
1. **Job counts:** one table that explains why different analyses use 351, 305, 294, 91, 53/45 or 42 jobs.
2. **AI/ML jobs subset** (decided 29 Sep): how many jobs are AI/ML jobs, and, *only if there are at least 30 usable ones* (rule fixed here, before counting), how the methods do on them.
3. **Speed on a plain CPU:** how long E5 takes to rank candidates for one job on a normal server without a GPU, the realistic case for a small job portal.
4. **Personal-data check** for the Ethics section: counts of emails, phone numbers and web links in the resumes (counts only, nothing is printed).

If Notebook 8c has not finished yet, its parts are skipped; run this notebook again afterwards.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U sentence-transformers

import os, re, json, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
OUT = f'{RESULTS}/paper'
os.makedirs(f'{OUT}/tables', exist_ok=True)
os.makedirs(f'{OUT}/figures', exist_ok=True)
SEED = 42
NUM = {}   # every number the paper quotes

def R(name, **kw):
    path = f'{RESULTS}/{name}'
    return pd.read_csv(path, **kw) if name.endswith('.csv') else pd.read_parquet(path)
def have(name):
    return os.path.exists(f'{RESULTS}/{name}')
def ci(s):
    a = re.findall(r'-?\d+\.?\d*(?:e-?\d+)?', str(s))
    return (float(a[0]), float(a[1])) if len(a) >= 2 else (np.nan, np.nan)
def save_table(df, name, caption):
    df.to_csv(f'{OUT}/tables/{name}.csv', index=False)
    with open(f'{OUT}/tables/{name}.tex', 'w') as f:
        f.write(df.to_latex(index=False, escape=True, caption=caption, label=f'tab:{name}', float_format='%.3f'))
    print(f'saved table {name} ({len(df)} rows)')
def save_fig(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(f'{OUT}/figures/{name}.{ext}', bbox_inches='tight', dpi=200)
    plt.show()
    print('saved figure', name)

plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 110})
ACCENT, GREY, DARK = '#2a6f97', '#9aa5b1', '#1f2933'

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet').assign(split=s) for s in ('train', 'val', 'test')],
                   ignore_index=True)
g = all_df.groupby('job_id').agg(candidates=('resume_id', 'size'),
                                 relevant=('relevance', lambda r: int((r > 0).sum())),
                                 levels=('relevance', 'nunique'))
usable = set(g[(g.candidates >= 5) & (g.relevant >= 1) & (g.levels >= 2)].index)
print(f'{len(all_df)} pairs, {all_df.job_id.nunique()} jobs, {all_df.resume_id.nunique()} resumes, {len(usable)} usable jobs')

## 1. Dataset: Table 1, Table 1a (job counts), Figure 1 (label composition)

In [ ]:
words = lambda s: s.str.split().str.len()
jobs_u, res_u = all_df.drop_duplicates('job_id'), all_df.drop_duplicates('resume_id')
jw, rw = words(jobs_u.job_text), words(res_u.resume_text)
lab = all_df.label.value_counts()
t1 = pd.DataFrame([
    ('Pairs', f'{len(all_df):,}'),
    ('Distinct jobs / resumes', f'{len(jobs_u):,} / {len(res_u):,}'),
    ('Labels: No Fit / Potential Fit / Good Fit', f"{lab['No Fit']:,} / {lab['Potential Fit']:,} / {lab['Good Fit']:,}"),
    ('Pairs per job, median', f"{all_df.groupby('job_id').size().median():.0f}"),
    ('Pairs per resume, median', f"{all_df.groupby('resume_id').size().median():.0f}"),
    ('Job length in words, median (Q1-Q3)', f'{jw.median():.0f} ({jw.quantile(.25):.0f}-{jw.quantile(.75):.0f})'),
    ('Resume length in words, median (Q1-Q3)', f'{rw.median():.0f} ({rw.quantile(.25):.0f}-{rw.quantile(.75):.0f})'),
    ('Longest resume (words)', f'{rw.max():,}'),
], columns=['', 'Value'])
save_table(t1, 't1_dataset', 'Dataset after cleaning.')
display(t1)

# Per-job label composition
def comp(r):
    fit = r[r > 0]
    if len(fit) == 0:
        return 'no fit candidate'
    return 'only Potential' if (fit == 1).all() else 'only Good' if (fit == 2).all() else 'both Good and Potential'
cat = all_df.groupby('job_id').relevance.agg(comp).value_counts()
order = ['only Potential', 'only Good', 'both Good and Potential', 'no fit candidate']
cat = cat.reindex(order, fill_value=0)
fig, ax = plt.subplots(figsize=(5.5, 1.9))
colors = [GREY, GREY, ACCENT, '#d9dee3']
ax.barh(order[::-1], cat[order[::-1]], color=colors[::-1])
for i, v in enumerate(cat[order[::-1]]):
    ax.text(v + 3, i, f'{v} ({v / cat.sum():.0%})', va='center')
ax.set_xlabel('jobs'); ax.set_xlim(0, cat.max() * 1.3); ax.grid(axis='y', visible=False)
ax.set_title('Which fit labels does each job use?', loc='left', fontsize=9)
save_fig(fig, 'fig1_label_composition')

key = pd.read_csv(f'{DATA}/human_labelling_key.csv')
test = all_df[all_df.split == 'test']
fit_jobs = int((cat.sum() - cat['no fit candidate']))
t1a = pd.DataFrame([
    (all_df.job_id.nunique(), 'all jobs', 'Table 1'),
    (fit_jobs, 'at least one fit candidate', 'label structure (6.3)'),
    (len(usable), 'usable: 5+ candidates, 1+ fit, 2+ labels', 'zero-shot comparison (6.1), AI polish (6.4)'),
    (int(cat['both Good and Potential']), 'both Good and Potential among fit candidates', 'per-job Good-vs-Potential AUC (6.3)'),
    (f"{test.job_id.nunique()} / {len(set(test.job_id) & usable)}", 'test split: all / usable', 'fine-tuned models, LLM judge (6.1-6.2)'),
    (key.job_id.nunique(), 'jobs in the 150-pair human sample', 'human relabelling (6.3)'),
], columns=['Jobs', 'Definition', 'Used in'])
if have('06a_cv_design.csv'):
    t1a.loc[len(t1a)] = (int(R('06a_cv_design.csv')['test jobs'].sum()), 'test cells of the 5x5 cross-validation grid', 'memorisation (6.1)')
save_table(t1a, 't1a_job_counts', 'Job counts used in the paper.')
display(t1a)

NUM['dataset'] = {'pairs': len(all_df), 'jobs': int(len(jobs_u)), 'resumes': int(len(res_u)),
                  'labels': {k: int(v) for k, v in lab.items()}, 'usable_jobs': len(usable),
                  'job_composition': {k: int(v) for k, v in cat.items()},
                  'resume_words_median': float(rw.median()), 'job_words_median': float(jw.median()),
                  'test_resumes_also_in_train': int(len(set(test.resume_id) & set(all_df[all_df.split == 'train'].resume_id))),
                  'test_resumes': int(test.resume_id.nunique())}
print(NUM['dataset'])

## 2. Results 1: keywords vs embeddings, all 294 usable jobs (Table 3)

In [ ]:
s3 = R('03b_all_jobs_summary.csv')
sig = R('03b_significance_vs_tfidf.csv')
sig_n = sig[sig.metric == 'NDCG@10'].set_index('method')
t3 = s3[['method', 'NDCG@10', 'NDCG@10 CI', 'AUC fit-vs-no', 'AUC good-vs-potential']].copy()
t3['diff vs TF-IDF'] = t3.method.map(sig_n['diff vs TF-IDF'])
t3['p (Holm)'] = t3.method.map(sig_n['p (Holm)'])
MAIN = ['E5-base (zero-shot)', 'MPNet-base (zero-shot)', 'BGE-base (zero-shot)', 'Nomic-v1.5 (zero-shot, long)',
        'TF-IDF', 'BM25', 'Resume-prior (no job text)', 'Random']
save_table(t3[t3.method.isin(MAIN)].round(4), 't3_zero_shot_main', 'Zero-shot ranking on all 294 usable jobs.')
save_table(t3.round(4), 't3_zero_shot_full', 'All zero-shot variants on all 294 usable jobs (appendix).')
display(t3.round(4))
NUM['zero_shot'] = t3.set_index('method')[['NDCG@10', 'AUC fit-vs-no', 'AUC good-vs-potential', 'p (Holm)']].round(4).to_dict('index')

## 3. Results 1: fine-tuning vs memorisation (Table 4, Figure 2)

In [ ]:
s6 = R('06a_cv_summary.csv')
t4 = s6[['method', 'AUC fit-vs-no', 'AUC fit-vs-no CI', 'AUC good-vs-potential', 'mean job AUC good-vs-potential', 'NDCG@5']]
save_table(t4.round(4), 't4_memorisation', 'Fine-tuned BGE under the standard, size-matched and strict training sets (5x5 cross-validation).')
display(t4.round(4))
tests = R('06a_cv_tests.csv')
nd5 = R('06c_ndcg5_tests.csv')
save_table(tests, 't4b_memorisation_tests', 'Paired differences, AUC (job-level bootstrap).')
save_table(nd5[['comparison', 'mean diff', 'CI', 'p Holm (Wilcoxon)']], 't4c_ndcg5_tests', 'Paired differences, NDCG@5.')
display(tests); display(nd5)

show = ['FT standard', 'FT size-matched', 'FT strict', 'BGE-base (zero-shot)', 'E5-base (zero-shot)', 'TF-IDF']
fig, axes = plt.subplots(1, 2, figsize=(7, 2.4), sharey=True)
for ax, col, title in [(axes[0], 'AUC fit-vs-no', 'Fit vs no fit'), (axes[1], 'AUC good-vs-potential', 'Good vs Potential')]:
    x = s6.set_index('method').loc[show]
    lo_hi = np.array([ci(v) for v in x[col + ' CI']])
    y = np.arange(len(show))[::-1]
    colors = [ACCENT if m.startswith('FT') else GREY for m in show]
    ax.errorbar(x[col], y, xerr=[x[col] - lo_hi[:, 0], lo_hi[:, 1] - x[col]], fmt='none', ecolor=DARK, elinewidth=1, capsize=2)
    ax.scatter(x[col], y, c=colors, zorder=3, s=28)
    ax.set_yticks(y); ax.set_yticklabels(show); ax.set_title(title, loc='left', fontsize=9); ax.set_xlabel('AUC (pooled)')
    if col == 'AUC good-vs-potential':
        ax.axvline(0.5, color=DARK, lw=0.8, ls=':')
# no suptitle: it collided with the panel titles; the caption carries the message
save_fig(fig, 'fig2_memorisation')
NUM['memorisation'] = {'summary': s6.set_index('method')[['AUC fit-vs-no', 'mean job AUC good-vs-potential', 'NDCG@5']].round(4).to_dict('index'),
                       'auc_tests': tests.to_dict('records'), 'ndcg5_tests': nd5[['comparison', 'mean diff', 'CI', 'p Holm (Wilcoxon)']].to_dict('records')}

## 4. Results 2: LLM judge, accuracy and cost (Table 5, Figure 3)

Cost is the median time to rank one test job on a T4 GPU, as measured in Notebooks 2–4. For the judge on all candidates, it is the time per pair times the median number of candidates in a usable test job. Macro-F1 is computed here from the saved pair scores (test pairs in usable jobs).

In [ ]:
from sklearn.metrics import f1_score
s4 = R('04_llm_summary.csv'); s4 = s4[s4.split == 'test']
cost = R('04_llm_cost.csv').set_index('model')
s2 = R('02_baselines_summary.csv'); s2 = s2[s2.split == 'test']
s3e = R('03_embeddings_summary.csv'); s3e = s3e[s3e.split == 'test']
ps = R('04_llm_pair_scores.parquet'); pt = ps[ps.split == 'test']
med_cand = pt.groupby('job_id').size().median()

f1 = {}
for col, name in (('qwen4b_label', 'Qwen3-4B judge (all pairs)'), ('qwen1_7b_label', 'Qwen3-1.7B judge (all pairs)')):
    x = pt.dropna(subset=[col])
    f1[name] = f1_score(x.label, x[col], average='macro')
print('macro-F1 (dataset labels):', {k: round(v, 4) for k, v in f1.items()}, '| Jin 2025: 0.450')

rows = []
for _, r in s2.iterrows():
    rows.append((r.method, r['NDCG@10'], r['ms/job (median)'], np.nan))
for m in ['E5-base (zero-shot)', 'BGE-base (zero-shot)', 'MPNet-base (zero-shot)', 'BGE-base (fine-tuned)', 'BGE-base (fine-tuned, strict)']:
    r = s3e[s3e.method == m].iloc[0]
    rows.append((m, r['NDCG@10'], r['ms/job (median)'], np.nan))
for m, model in [('Qwen3-4B judge (all pairs)', 'Qwen3-4B'), ('Qwen3-1.7B judge (all pairs)', 'Qwen3-1.7B')]:
    rows.append((m, s4.set_index('method').loc[m, 'NDCG@10'], cost.loc[model, 'ms/pair (median)'] * med_cand, f1[m]))
for m, model in [('E5 -> Qwen3-4B re-rank top-10', 'Qwen3-4B'), ('E5 -> Qwen3-1.7B re-rank top-10', 'Qwen3-1.7B')]:
    rows.append((m, s4.set_index('method').loc[m, 'NDCG@10'], cost.loc[model, 'ms/job for top-10 re-rank'], np.nan))
t5 = pd.DataFrame(rows, columns=['method', 'NDCG@10 (test)', 'ms per job (T4)', 'macro-F1'])
t5['seconds per job'] = (t5['ms per job (T4)'] / 1000).round(3)
save_table(t5.round(4), 't5_llm_cost', f'Test-split ranking quality and cost per job (median {med_cand:.0f} candidates).')
display(t5.round(4))

# hand-placed labels for the crowded 0.02-0.1 s cluster (they overlapped in the first run)
LABEL_POS = {'BGE-base (fine-tuned, strict)': (-8, 13, 'right'), 'MPNet-base': (8, 13, 'left'),
             'E5-base': (-8, 0, 'right'), 'BGE-base': (-8, -11, 'right'), 'TF-IDF': (8, -9, 'left'),
             'E5 -> Qwen3-1.7B re-rank top-10': (-6, 0, 'right'), 'Qwen3-1.7B judge': (5, -9, 'left')}
fig, ax = plt.subplots(figsize=(5.5, 3.2))
for _, r in t5.iterrows():
    llm = 'Qwen' in r.method
    ax.scatter(max(r['seconds per job'], 1e-4), r['NDCG@10 (test)'], color=ACCENT if llm else GREY, s=26, zorder=3)
    label = r.method.replace(' (zero-shot)', '').replace(' (all pairs)', '')
    dx, dy, ha = LABEL_POS.get(label, (3, 2, 'left'))
    ax.annotate(label, (max(r['seconds per job'], 1e-4), r['NDCG@10 (test)']), fontsize=6.5, ha=ha, va='center',
                xytext=(dx, dy), textcoords='offset points',
                arrowprops=dict(arrowstyle='-', color=GREY, lw=0.5) if abs(dy) > 5 else None)
ax.set_xscale('log'); ax.set_xlabel('seconds to rank one job (log scale, T4 GPU)'); ax.set_ylabel('NDCG@10, test')
ax.set_title('The LLM judge costs 100-1000x more time without a significant gain', loc='left', fontsize=9)
save_fig(fig, 'fig3_accuracy_vs_cost')
NUM['llm'] = {'table': t5.round(4).to_dict('records'), 'median_candidates_test': float(med_cand),
              'ms_per_pair': cost['ms/pair (median)'].to_dict(), 'macro_f1': {k: round(v, 4) for k, v in f1.items()}}

## 5. Speed on a plain CPU (the small-portal case)

A small job portal usually runs on a server without a GPU. Resumes are encoded once when they are uploaded; when a job is posted, only the job is encoded and compared with every stored resume. We time both on this Colab CPU with E5-base.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
m = SentenceTransformer('intfloat/e5-base-v2', device=dev); m.max_seq_length = 512
cut = lambda t: ' '.join(t.split()[:900])
sample_res = res_u.sample(100, random_state=SEED).resume_text.map(cut).tolist()
m.encode(['passage: warm up'] * 2)
t0 = time.perf_counter(); emb = m.encode(['passage: ' + t for t in sample_res], batch_size=16, normalize_embeddings=True)
index_ms = (time.perf_counter() - t0) * 1000 / len(sample_res)
pool = np.vstack([emb] * 7)[:len(res_u)]          # stand-in for all 643 stored resumes
q_ms = []
for t in jobs_u.job_text.sample(20, random_state=SEED):
    t0 = time.perf_counter()
    q = m.encode(['query: ' + t], normalize_embeddings=True)[0]
    order = np.argsort(-(pool @ q))
    q_ms.append((time.perf_counter() - t0) * 1000)
cpu = {'device': dev, 'cpu_threads': torch.get_num_threads(), 'index_ms_per_resume': round(index_ms, 1),
       'ms_to_rank_one_job_over_643_resumes_median': round(float(np.median(q_ms)), 1)}
print(cpu)
NUM['cpu_latency'] = cpu
del m

## 6. Results 3: what Good vs Potential measures; human labels (Tables 6–7, Figure 4)

In [ ]:
auc = R('05_auc_by_labels.csv')
gp = auc[auc.task == 'good vs potential']
t6 = gp.pivot(index='method', columns='labels', values='AUC').rename(columns={'dataset': 'AUC, dataset labels', 'you': 'AUC, human labels'})
t6['pairs (dataset / human)'] = gp.pivot(index='method', columns='labels', values='pairs').astype(int).astype(str).agg(' / '.join, axis=1)
METHODS = ['Resume-prior', 'TF-IDF', 'E5-base', 'BGE-base', 'MPNet-base', 'Qwen3-4B judge', 'Qwen3-1.7B judge']
t6 = t6.reindex(METHODS).reset_index()
shift = R('05_auc_shift.csv').set_index('method')
t6['shift (human - dataset)'] = t6.method.map(shift['AUC(you) - AUC(dataset)'])
t6['p'] = t6.method.map(shift['p (two-sided)'])
save_table(t6.round(3), 't6_good_vs_potential', 'Good-vs-Potential AUC on the 150 relabelled pairs: dataset labels vs human labels.')
display(t6.round(3))

agr = R('05_agreement.csv')
save_table(agr, 't7_agreement', 'Agreement between the human labels and the dataset labels (150 pairs).')
display(agr)
conf = R('05_confusion.csv')
save_table(conf.rename(columns={conf.columns[0]: ''}), 't7b_confusion', 'Confusion matrix: dataset labels (rows) vs human labels (columns).')
display(conf)
f1h = R('05_llm_f1.csv'); display(f1h)

fig, ax = plt.subplots(figsize=(5.5, 2.8))
y = np.arange(len(METHODS))[::-1]
for k, (labels, color, dy) in enumerate([('dataset', GREY, 0.18), ('you', ACCENT, -0.18)]):
    x = gp[gp.labels == labels].set_index('method').reindex(METHODS)
    lo_hi = np.array([ci(v) for v in x.CI])
    ax.errorbar(x.AUC, y + dy, xerr=[x.AUC - lo_hi[:, 0], lo_hi[:, 1] - x.AUC], fmt='o', color=color, ms=4,
                elinewidth=1, capsize=2, label='dataset labels' if labels == 'dataset' else 'human labels')
ax.axvline(0.5, color=DARK, lw=0.8, ls=':')
ax.set_yticks(y); ax.set_yticklabels(METHODS); ax.set_xlabel('AUC, Good vs Potential'); ax.legend(loc='lower right', fontsize=7)
ax.set_title('Job-aware methods agree with the human, not the dataset;\nthe resume-only prior does the opposite', loc='left', fontsize=9)
save_fig(fig, 'fig4_good_vs_potential')

va = R('06b_variance_decomposition.csv'); txt = R('06b_text_only_auc.csv'); tech = R('06c_tech_split_auc.csv')
display(va); display(txt); display(tech)
save_table(txt, 't6b_resume_text_only', 'Per-job Good-vs-Potential AUC of scorers that never read the job (91 mixed jobs).')
NUM['labels'] = {'agreement': agr.to_dict('records'), 'gp_auc': t6.round(4).to_dict('records'), 'llm_f1': f1h.to_dict('records'),
                 'variance': va.to_dict('records'), 'text_only': txt.to_dict('records'), 'tech_split': tech.to_dict('records'),
                 'purity': R('06b_purity.csv').to_dict('records')}

## 7. Results 4: AI-polished resumes (Table 8, Figure 5)

In [ ]:
pe = R('08_polish_effects.csv')
t8 = pe[['method', 'pairs', 'n', 'change in SDs', 'CI', 'share up', 'mean places up', 'share reaching top 5', 'p (Holm)']]
save_table(t8, 't8_polish', 'Score change when a resume is replaced by its AI-polished version (181 resumes).')
display(t8)
steps = R('08b_steps.csv')
STEP_ORDER = ['cleaning', 'markdown symbols', 'polish beyond cleaning', 'total']
extra = None
if have('08c_markdown_steps.csv'):
    extra = R('08c_markdown_steps.csv')
    display(extra)
    save_table(extra[['step', 'method', 'change in SDs', 'CI (SDs)', 'share up', 'mean places up', 'share reaching top 5', 'p (Holm)']],
               't8c_markdown_injection', 'Score change when markdown is added to the original resume text (no rewording).')
else:
    print('Notebook 8c results not found yet: Figure 5 is drawn without them. Run this notebook again after 8c.')
save_table(steps[['step', 'method', 'change in SDs', 'CI (SDs)', 'share up', 'mean places up', 'p (Holm)']],
           't8b_steps', 'Decomposing the polish effect: cleaning, markdown symbols and wording.')
display(steps)

meths = ['tfidf', 'e5', 'bge', 'mpnet', 'qwen4b judge']
names = {'tfidf': 'TF-IDF', 'e5': 'E5', 'bge': 'BGE', 'mpnet': 'MPNet', 'qwen4b judge': 'Qwen3-4B judge'}
bars = [('AI polish, total', steps, 'total'), ('restoring spaces only', steps, 'cleaning'),
        ('markdown in the AI text', steps, 'markdown symbols')]
if extra is not None:
    bars.append(('markdown added to original', extra, 'all formatting'))
fig, ax = plt.subplots(figsize=(6.5, 2.8))
w = 0.8 / len(bars); palette = [ACCENT, '#d9dee3', '#7fb3d5', '#e07a5f']
for k, (label, df, step) in enumerate(bars):
    x = df[df.step == step].set_index('method').reindex(meths)
    val = x['change in SDs'].to_numpy(dtype=float)
    lo_hi = np.array([ci(v) for v in x['CI (SDs)']])
    pos = np.arange(len(meths)) + (k - (len(bars) - 1) / 2) * w
    ax.bar(pos, val, w, color=palette[k], label=label)
    ok = ~np.isnan(val)
    ax.errorbar(pos[ok], val[ok], yerr=[val[ok] - lo_hi[ok, 0], lo_hi[ok, 1] - val[ok]], fmt='none', ecolor=DARK, elinewidth=0.8, capsize=1.5)
ax.axhline(0, color=DARK, lw=0.8)
ax.set_xticks(np.arange(len(meths))); ax.set_xticklabels([names[m] for m in meths])
ax.set_ylabel('score change (SDs)'); ax.legend(fontsize=7, frameon=False)
ax.set_title('E5 rewards markdown formatting, not cleaner text or fit', loc='left', fontsize=9)
save_fig(fig, 'fig5_polish')
NUM['polish'] = {'effects': t8.to_dict('records'), 'steps': steps.to_dict('records'),
                 'markdown_injection': extra.to_dict('records') if extra is not None else None,
                 'rewrites': R('08_rewrite_checks.csv')[['usable', 'strict']].sum().astype(int).to_dict()}

## 8. AI/ML jobs subset

**Rule fixed before counting:** a job is an AI/ML job if its text mentions at least one of: machine learning, deep learning, artificial intelligence, AI (as a word), data scientist, data science, NLP, natural language processing, computer vision, LLM, large language model, neural network, TensorFlow, PyTorch, scikit-learn. We report the subset only if there are **at least 30 usable AI/ML jobs**; otherwise the paper gives the count and says the subset is too small.

In [ ]:
AI_TERMS = r'machine learning|deep learning|artificial intelligence|\bAI\b|data scientist|data science|\bNLP\b|natural language processing|computer vision|\bLLMs?\b|large language model|neural network|tensorflow|pytorch|scikit-learn'
ai = jobs_u.set_index('job_id').job_text.str.contains(AI_TERMS, case=False, regex=True)
ai_jobs = set(ai[ai].index)
ai_usable = ai_jobs & usable
print(f'AI/ML jobs: {len(ai_jobs)} of {len(jobs_u)}; usable: {len(ai_usable)} of {len(usable)}')
NUM['ai_subset'] = {'ai_jobs': len(ai_jobs), 'ai_usable': len(ai_usable), 'reported': len(ai_usable) >= 30}

if len(ai_usable) >= 30:
    pq = R('03b_all_jobs_per_query.csv')
    pq['ai'] = pq.job_id.isin(ai_usable)
    sub = pq.groupby(['method', 'ai'])[['NDCG@10', 'AUC fit-vs-no', 'AUC good-vs-potential']].mean().unstack().round(4)
    display(sub)
    def perm_p(a, b, n=10000):
        rng = np.random.default_rng(SEED); d = np.asarray(a) - np.asarray(b)
        return float((np.abs((rng.choice([-1, 1], size=(n, len(d))) * d).mean(axis=1)) >= abs(d.mean())).mean())
    wide = pq[pq.ai].pivot(index='job_id', columns='method', values='NDCG@10')
    rows = []
    for mth in ['E5-base (zero-shot)', 'BGE-base (zero-shot)', 'MPNet-base (zero-shot)', 'Resume-prior (no job text)']:
        rows.append({'method': mth, 'jobs': len(wide), 'NDCG@10': round(wide[mth].mean(), 4),
                     'diff vs TF-IDF': round((wide[mth] - wide['TF-IDF']).mean(), 4), 'p': perm_p(wide[mth], wide['TF-IDF'])})
    ai_t = pd.DataFrame(rows); display(ai_t)
    save_table(ai_t, 't9_ai_jobs', f'Zero-shot ranking on the {len(ai_usable)} usable AI/ML jobs.')
    NUM['ai_subset']['table'] = ai_t.to_dict('records')
else:
    print('Fewer than 30 usable AI/ML jobs: the paper reports the count only.')

## 9. Personal-data check (for the Ethics section)

Counts only: how many resumes contain something that looks like an email address, a phone number or a web link. Nothing is printed.

In [ ]:
pii = {'email': r'[\w.+-]+@[\w-]+\.[\w.]+',
       'phone': r'(?:\+?\d{1,3}[\s.-]?)?\(?\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}',
       'web link': r'https?://|www\.|linkedin\.com'}
counts = {k: int(res_u.resume_text.str.contains(p, regex=True).sum()) for k, p in pii.items()}
print(f'resumes ({len(res_u)}) containing:', counts)
print('placeholder words such as "Company Name" / "City, State":',
      int(res_u.resume_text.str.contains('Company Name|City , State|City, State', regex=True).sum()))
NUM['pii_counts'] = counts

## 10. Save every number

In [ ]:
def clean(o):
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, float) and np.isnan(o):
        return None
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o
with open(f'{OUT}/numbers.json', 'w') as f:
    json.dump(clean(NUM), f, indent=1)
print('saved', f'{OUT}/numbers.json')
print('tables:', sorted(os.listdir(f'{OUT}/tables')))
print('figures:', sorted(os.listdir(f'{OUT}/figures')))
print('Next: download this notebook (File > Download > .ipynb).')